In [0]:
%pip install google-cloud-bigquery
%pip install db-dtypes
%pip install "pandas-gbq>=0.26.1" google-cloud-bigquery-storage

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pandas-gbq]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import json

credential_json = dbutils.secrets.get(
    scope="gcp-bigquery",
    key="service-account-json"
)

assert credential_json is not None
assert len(credential_json) > 0

print("GCP credential loaded successfully.")



GCP credential loaded successfully.


In [0]:
from google.cloud import bigquery
from google.oauth2 import service_account

print("BigQuery client library available.")

BigQuery client library available.


In [0]:
# Service Account Credential 객체 생성
service_account_info = json.loads(
    credential_json
)

credentials = (
    service_account.Credentials
    .from_service_account_info(
        service_account_info
    )
)

PROJECT_ID = "nimble-crane-431005-f8"
DATASET_ID = "clinical_trials"

client = bigquery.Client(
    project=PROJECT_ID,
    credentials=credentials
)

In [0]:
query_job = client.query(
    "SELECT 1 AS connection_test"
)

result = list(query_job.result())

print(result[0]["connection_test"])

1


In [0]:
gold_tables = [
    "gold_yearly_summary",
    "gold_country_summary",
    "gold_condition_trends"
]

for table_name in gold_tables:
    df = spark.read.table(
        f"clinical_trials.{table_name}"
    )

    print(
        table_name,
        "rows =",
        df.count()
    )

gold_yearly_summary rows = 1105
gold_country_summary rows = 1785
gold_condition_trends rows = 19561


In [0]:
# Bronze/Silver 수십만 건을 이런 식으로 toPandas() 하면 안 됩니다. 모든 데이터를 driver 메모리로 가져오기 때문입니다.
gold_yearly_spark_df = spark.read.table(
    "clinical_trials.gold_yearly_summary"
)

gold_yearly_pdf = gold_yearly_spark_df.toPandas()

print(gold_yearly_pdf.shape)

gold_yearly_pdf.head()

(1105, 7)


,study_year,study_type,phase_label,overall_status,study_count,total_enrollment,avg_enrollment
0,1966,INTERVENTIONAL,PHASE3,COMPLETED,1,NaN,NaN
1,1971,OBSERVATIONAL,NOT_SPECIFIED,COMPLETED,1,2657.0,2657.0
2,1976,INTERVENTIONAL,PHASE3,COMPLETED,1,NaN,NaN
3,1977,OBSERVATIONAL,NOT_SPECIFIED,COMPLETED,1,35002.0,35002.0
4,1981,OBSERVATIONAL,NOT_SPECIFIED,TERMINATED,1,597.0,597.0


In [0]:
# BigQuery Table ID 지
TABLE_ID = (
    f"{PROJECT_ID}."
    f"{DATASET_ID}."
    f"gold_yearly_summary"
)

print(TABLE_ID)

nimble-crane-431005-f8.clinical_trials.gold_yearly_summary


In [0]:
# BigQuery 적재
# Gold Dataset은 현재 분석용 snapshot을 제공하는 테이블이므로
# BigQuery 적재 시 WRITE_TRUNCATE를 사용해 기존 데이터를 최신 Gold 결과로 교체한다.


job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

load_job = client.load_table_from_dataframe(
    gold_yearly_pdf,
    TABLE_ID,
    job_config=job_config
)

load_job.result()

print("BigQuery load completed.")

/local_disk0/.ephemeral_nfs/envs/pythonEnv-f17f2c12-a3ef-424b-8235-ada6c4adabf3/lib/python3.12/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


BigQuery load completed.


In [0]:
# 적재 결과 확인
# gold_yearly_spark_df.count() 와 table.num_rows가 같아야 함
table = client.get_table(TABLE_ID)

print("Table:", table.full_table_id)
print("Rows :", table.num_rows)

Table: nimble-crane-431005-f8:clinical_trials.gold_yearly_summary
Rows : 1105


In [0]:
# BigQuery SQL 실제 조회
query = f"""
SELECT
    study_year,
    SUM(study_count) AS trial_count
FROM `{PROJECT_ID}.{DATASET_ID}.gold_yearly_summary`
GROUP BY study_year
ORDER BY study_year DESC
LIMIT 20
"""

result_df = client.query(
    query
).to_dataframe()

display(result_df)

/local_disk0/.ephemeral_nfs/envs/pythonEnv-f17f2c12-a3ef-424b-8235-ada6c4adabf3/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


study_year,trial_count
2030,1
2028,7
2027,79
2026,1846
2025,1173
2024,973
2023,852
2022,801
2021,751
2020,624


In [0]:
# 나머지 적재

def export_gold_to_bigquery(
    source_table_name,
    target_table_name=None
):
    if target_table_name is None:
        target_table_name = source_table_name

    spark_df = spark.read.table(
        f"clinical_trials.{source_table_name}"
    )

    source_count = spark_df.count()

    pandas_df = spark_df.toPandas()

    table_id = (
        f"{PROJECT_ID}."
        f"{DATASET_ID}."
        f"{target_table_name}"
    )

    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_TRUNCATE"
    )

    load_job = client.load_table_from_dataframe(
        pandas_df,
        table_id,
        job_config=job_config
    )

    load_job.result()

    target_table = client.get_table(
        table_id
    )

    target_count = target_table.num_rows

    print(
        f"{source_table_name}: "
        f"source={source_count}, "
        f"target={target_count}"
    )

    if source_count != target_count:
        raise ValueError(
            f"Row count mismatch: "
            f"{source_count} != {target_count}"
        )

In [0]:
for table_name in [
    "gold_yearly_summary",
    "gold_country_summary",
    "gold_condition_trends"
]:
    export_gold_to_bigquery(
        table_name
    )

/local_disk0/.ephemeral_nfs/envs/pythonEnv-f17f2c12-a3ef-424b-8235-ada6c4adabf3/lib/python3.12/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


gold_yearly_summary: source=1105, target=1105


/local_disk0/.ephemeral_nfs/envs/pythonEnv-f17f2c12-a3ef-424b-8235-ada6c4adabf3/lib/python3.12/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


gold_country_summary: source=1785, target=1785


/local_disk0/.ephemeral_nfs/envs/pythonEnv-f17f2c12-a3ef-424b-8235-ada6c4adabf3/lib/python3.12/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


gold_condition_trends: source=19561, target=19561
